In [ ]:
# ============================================================================
# STAGE 6c -- CONSENSUS vs ALTERNATIVE vs RAW-COT-TRACE  (matplotlib)
#
# Three PNGs per question:
#   <qid>_consensus.png   : highest-Phi proof subgraph of consensus answer
#   <qid>_alternative.png : same conclusion, argmin_S phi(S) at every Or node
#   <qid>_rawcot.png      : a random raw CoT trace (Stage 1 + Stage 2) that
#                             reached the correct answer, drawn as its DAG
#
# Inputs:
#   <DAG_DIR>/*_parsed_<scheme>.jsonl                  (Stage 5)
#   <DAG_DIR>/*_ensemble_dags_weighted_<scheme>.jsonl  (Stage 4)
#   <DAG_DIR>/*_eval_cots.csv                          (Stage 1)
#   <DAG_DIR>/*_eval_dags.jsonl                        (Stage 2)
# Outputs:
#   <PNG_DIR>/<scheme>/<dataset>/<qid>_consensus.png
#   <PNG_DIR>/<scheme>/<dataset>/<qid>_alternative.png
#   <PNG_DIR>/<scheme>/<dataset>/<qid>_rawcot.png
#   <PNG_DIR>/<scheme>/<dataset>/<qid>_meta.json
# ============================================================================

import csv, glob, json, os, random, sys, textwrap
from collections import defaultdict
from typing import Dict, List, Optional, Set, FrozenSet, Tuple

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

csv.field_size_limit(sys.maxsize)

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
DAG_DIR = "./outputs_4234"
PNG_DIR = os.path.join(DAG_DIR, "consensus_vs_alternative_vs_rawcot_pngs")
os.makedirs(PNG_DIR, exist_ok=True)

SCHEME_TAGS             = ["simple"]
N_QUESTIONS_PER_DATASET = 20
SAMPLE_SEED             = 7777

ALT_STRATEGY            = "min_phi"
MIN_NODE_SET_DIFF       = 1
N_RANDOM_FALLBACK_TRIES = 200

TYPE_COLORS = {
    "Planning":    "#cde7ff",
    "Fact":        "#d6f5d6",
    "Reasoning":   "#ffe9c7",
    "Conclusion":  "#f3d4ff",
    "Event":       "#d6f5d6",
    "Rule":        "#ffe9c7",
    "Implication": "#cde7ff",
    "Answer":      "#ffd6d6",
}
DEFAULT_COLOR = "#e8e8e8"

WRAP_WIDTH       = 22
NODE_WIDTH       = 2.6
NODE_HEIGHT_BASE = 0.65
LINE_HEIGHT      = 0.22
COL_SPACING      = 3.2
ROW_SPACING      = 3.0
FONT_SIZE        = 8
TITLE_FONT_SIZE  = 11
DPI              = 150


# ============================================================================
# 1. DATA LOADERS
# ============================================================================
def load_weighted_index(parsed_path: str, scheme: str) -> Dict[str, Dict]:
    dataset = os.path.basename(parsed_path).split("_parsed_")[0]
    p = os.path.join(os.path.dirname(parsed_path),
                     f"{dataset}_ensemble_dags_weighted_{scheme}.jsonl")
    idx: Dict[str, Dict] = {}
    if not os.path.exists(p):
        return idx
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            qid = r.get("question_id")
            if qid is not None:
                idx[qid] = r
    return idx


def load_correct_consensus_records(parsed_path: str) -> List[Dict]:
    out = []
    if not os.path.exists(parsed_path):
        return out
    with open(parsed_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            gold = (rec.get("gold_label") or "").strip().lower()
            cons = (rec.get("consensus_answer") or "").strip().lower()
            if gold and cons and gold == cons:
                out.append(rec)
    return out


def consensus_graph_from_record(rec: Dict) -> Optional[Dict]:
    consensus = rec.get("consensus_answer")
    for pc in rec.get("per_conclusion", []) or []:
        if pc.get("answer_text") == consensus and pc.get("topk"):
            return pc["topk"][0]
    return None


def consensus_cluster_id(rec: Dict) -> Optional[str]:
    consensus = rec.get("consensus_answer")
    for pc in rec.get("per_conclusion", []) or []:
        if pc.get("answer_text") == consensus:
            return pc.get("answer_cluster_id")
    return None


# ----------------------------------------------------------------------------
# Raw CoT lookup: pair each (qid, model, sample_idx) trace from the eval CSV
# with the corresponding extracted DAG from <ds>_eval_dags.jsonl.
#
# Returns:  question_id -> list of {model, sample_idx, predicted_label, dag}
#           where dag is the Stage-2 parsed structure (only parse_ok records).
# ----------------------------------------------------------------------------
def load_raw_cot_index(dataset: str) -> Dict[str, List[Dict]]:
    csv_path  = os.path.join(DAG_DIR, f"{dataset}_eval_cots.csv")
    jsonl_path = os.path.join(DAG_DIR, f"{dataset}_eval_dags.jsonl")
    if not (os.path.exists(csv_path) and os.path.exists(jsonl_path)):
        return {}

    # Build (qid, model, sample_idx) -> dag, and pull the predicted label
    # from the DAG record itself (Stage 2 stores it).
    key_to_dag: Dict[Tuple[str, str, int], Dict] = {}
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not r.get("parse_ok"):
                continue
            if r.get("dag") is None:
                continue
            try:
                key = (r["question_id"], r["model"], int(r["sample_idx"]))
            except (KeyError, ValueError, TypeError):
                continue
            key_to_dag[key] = {
                "dag":             r["dag"],
                "predicted_label": r.get("predicted_label", ""),
                "model":           r["model"],
                "sample_idx":      int(r["sample_idx"]),
            }

    # Walk the CSV to enumerate every trace per question_id, joining on key.
    # We only really need the CSV to know which traces exist per question,
    # but joining ensures we never include a DAG without a CSV-attested trace.
    by_qid: Dict[str, List[Dict]] = defaultdict(list)
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            qid    = (row.get("question_id")   or "").strip()
            model  = (row.get("model")          or "").strip()
            try:
                samp = int(row.get("sample_idx") or "")
            except ValueError:
                continue
            if not qid or not model:
                continue
            entry = key_to_dag.get((qid, model, samp))
            if entry is None:
                continue   # DAG missing or failed validation -- skip
            by_qid[qid].append(entry)
    return dict(by_qid)


# ----------------------------------------------------------------------------
# Convert a Stage-2 DAG dict ({"nodes": [...]} where each node has support
# bundles) into the shape the renderer expects ({nodes, edges, node_details}).
# Edges are derived from support bundles: [child, parent] per bundle member.
# ----------------------------------------------------------------------------
def normalize_per_trace_dag(dag: Dict) -> Optional[Dict]:
    nodes_raw = (dag or {}).get("nodes") or []
    if not nodes_raw:
        return None
    id_to_node: Dict[str, Dict] = {}
    for n in nodes_raw:
        if isinstance(n, dict) and isinstance(n.get("id"), str):
            id_to_node[n["id"]] = n
    if not id_to_node:
        return None
    edges: List[Tuple[str, str]] = []
    for nid, node in id_to_node.items():
        for bundle in node.get("support", []) or []:
            if not isinstance(bundle, list):
                continue
            for m in bundle:
                if m in id_to_node:
                    edges.append((nid, m))
    node_ids = list(id_to_node.keys())
    return {
        "nodes": sorted(node_ids),
        "edges": [[c, p] for (c, p) in edges],
        "node_details": [{
            "id":   nid,
            "type": id_to_node[nid].get("type"),
            "text": id_to_node[nid].get("text"),
        } for nid in node_ids],
    }


def pick_random_correct_cot(traces_for_qid: List[Dict],
                             gold_answer: str,
                             rng: random.Random) -> Optional[Dict]:
    """Choose one trace whose predicted_label matches the gold answer,
    return its normalized DAG plus provenance fields."""
    want = (gold_answer or "").strip().lower()
    matching = [t for t in traces_for_qid
                if (t.get("predicted_label") or "").strip().lower() == want]
    if not matching:
        return None
    chosen = rng.choice(matching)
    g = normalize_per_trace_dag(chosen["dag"])
    if g is None:
        return None
    g["_source_model"] = chosen["model"]
    g["_sample_idx"]   = chosen["sample_idx"]
    return g


# ============================================================================
# 2. ALTERNATIVE GRAPH UNFOLDING  (unchanged)
# ============================================================================
def attach_details(graph_nodes, graph_edges, id_to_node):
    return {
        "nodes": sorted(graph_nodes),
        "edges": [[c, p] for (c, p) in graph_edges],
        "node_details": [{
            "id":   nid,
            "type": id_to_node.get(nid, {}).get("type"),
            "text": id_to_node.get(nid, {}).get("text"),
        } for nid in graph_nodes],
    }


def phi_of_bundle(bundle, W):
    members = bundle.get("members", [])
    return min(W.get(m, 0.0) for m in members) if members else 0.0


def unfold_alternative(target, id_to_node, W, strategy, rng):
    nodes: Set[str] = set()
    edges: List[Tuple[str, str]] = []

    def unfold(nid, on_path):
        if nid in on_path:
            nodes.add(nid); return
        node = id_to_node.get(nid)
        if node is None:
            nodes.add(nid); return
        nodes.add(nid)
        gate = node.get("gate", "Atomic")
        support = node.get("support", [])
        if gate == "Atomic" or not support:
            return
        if gate == "And":
            chosen = support[0]
        elif strategy == "min_phi":
            chosen = min(support, key=lambda S: phi_of_bundle(S, W))
        else:
            chosen = support[rng.randrange(len(support))]
        new_path = on_path | {nid}
        for m in chosen.get("members", []):
            edges.append((nid, m))
            unfold(m, new_path)

    unfold(target, frozenset())
    return attach_details(sorted(nodes), edges, id_to_node)


def get_alternative_graph(target, id_to_node, W, cg_nodeset, strategy, rng):
    alt = unfold_alternative(target, id_to_node, W, strategy, rng)
    if len(frozenset(alt["nodes"]).symmetric_difference(cg_nodeset)) \
            >= MIN_NODE_SET_DIFF:
        return alt
    for _ in range(N_RANDOM_FALLBACK_TRIES):
        alt = unfold_alternative(target, id_to_node, W, "random", rng)
        if len(frozenset(alt["nodes"]).symmetric_difference(cg_nodeset)) \
                >= MIN_NODE_SET_DIFF:
            return alt
    return None


# ============================================================================
# 3. LAYOUT + RENDER  (unchanged)
# ============================================================================
def hierarchical_layout(graph: Dict) -> Dict[str, Tuple[float, float]]:
    node_ids = list(graph["nodes"])
    idset = set(node_ids)

    children: Dict[str, List[str]] = defaultdict(list)
    preds: Dict[str, List[str]]    = defaultdict(list)
    for edge in graph.get("edges", []) or []:
        if len(edge) != 2:
            continue
        c, p = edge
        if c in idset and p in idset:
            children[p].append(c)
            preds[c].append(p)

    layer: Dict[str, int] = {}
    def get_layer(n: str, on_path: FrozenSet[str]) -> int:
        if n in layer:
            return layer[n]
        if n in on_path:
            layer[n] = 0
            return 0
        if not preds[n]:
            layer[n] = 0
            return 0
        new_path = on_path | {n}
        ll = 1 + max(get_layer(p, new_path) for p in preds[n])
        layer[n] = ll
        return ll
    for n in node_ids:
        get_layer(n, frozenset())

    by_layer: Dict[int, List[str]] = defaultdict(list)
    for n in node_ids:
        by_layer[layer[n]].append(n)

    pos: Dict[str, Tuple[float, float]] = {}
    max_layer = max(by_layer.keys()) if by_layer else 0
    for ly in range(max_layer + 1):
        nodes_in_layer = by_layer[ly]
        if ly == 0:
            nodes_sorted = sorted(nodes_in_layer)
        else:
            def avg_pred_x(n):
                xs = [pos[p][0] for p in preds[n] if p in pos]
                return sum(xs) / len(xs) if xs else 0.0
            nodes_sorted = sorted(nodes_in_layer, key=avg_pred_x)
        n_layer = len(nodes_sorted)
        total_w = (n_layer - 1) * COL_SPACING
        x_start = -total_w / 2.0
        for i, n in enumerate(nodes_sorted):
            pos[n] = (x_start + i * COL_SPACING, ly * ROW_SPACING)

    return pos


def render_graph(graph: Dict, title_label: str, out_path: str) -> bool:
    node_ids = list(graph.get("nodes", []))
    if not node_ids:
        return False
    details = {d["id"]: d for d in (graph.get("node_details") or [])}

    pos = hierarchical_layout(graph)
    if not pos:
        return False

    wrapped: Dict[str, List[str]] = {}
    heights: Dict[str, float]     = {}
    for nid in node_ids:
        d = details.get(nid, {})
        ntype = d.get("type") or "?"
        ntext = d.get("text") or nid
        lines = [f"[{ntype}]"] + (textwrap.wrap(str(ntext), WRAP_WIDTH)
                                  or [str(nid)])
        wrapped[nid] = lines
        heights[nid] = NODE_HEIGHT_BASE + LINE_HEIGHT * max(0, len(lines) - 1)

    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    x_min, x_max = min(xs) - NODE_WIDTH, max(xs) + NODE_WIDTH
    y_min = min(ys) - max(heights.values())
    y_max = max(ys) + max(heights.values()) + 1.5
    fig_w = max(8.0, (x_max - x_min) * 0.6)
    fig_h = max(5.0, (y_max - y_min) * 0.55)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=DPI)
    ax.set_xlim(x_min - 0.5, x_max + 0.5)
    ax.set_ylim(y_min - 0.5, y_max + 0.5)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_facecolor("white")

    ax.text((x_min + x_max) / 2, y_max - 0.3, title_label,
            ha="center", va="top",
            fontsize=TITLE_FONT_SIZE, fontweight="bold",
            family="DejaVu Sans")

    for nid in node_ids:
        x, y = pos[nid]
        h = heights[nid]
        d = details.get(nid, {})
        fill = TYPE_COLORS.get(d.get("type") or "?", DEFAULT_COLOR)
        box = FancyBboxPatch(
            (x - NODE_WIDTH / 2, y - h / 2), NODE_WIDTH, h,
            boxstyle="round,pad=0.02,rounding_size=0.12",
            linewidth=1.2, edgecolor="#555555", facecolor=fill,
            zorder=2,
        )
        ax.add_patch(box)
        lines = wrapped[nid]
        n_lines = len(lines)
        line_y0 = y + (n_lines - 1) * LINE_HEIGHT / 2
        for i, line in enumerate(lines):
            weight = "bold" if i == 0 else "normal"
            ax.text(x, line_y0 - i * LINE_HEIGHT, line,
                    ha="center", va="center",
                    fontsize=FONT_SIZE, fontweight=weight,
                    family="DejaVu Sans", zorder=3)

    for edge in graph.get("edges", []) or []:
        if len(edge) != 2:
            continue
        c, p = edge
        if c not in pos or p not in pos:
            continue
        x_p, y_p = pos[p]
        x_c, y_c = pos[c]
        h_p = heights[p] / 2 + 0.04
        h_c = heights[c] / 2 + 0.04
        if y_c > y_p:
            y_p_edge = y_p + h_p
            y_c_edge = y_c - h_c
        else:
            y_p_edge = y_p - h_p
            y_c_edge = y_c + h_c

        arrow = FancyArrowPatch(
            (x_p, y_p_edge), (x_c, y_c_edge),
            arrowstyle="-|>", mutation_scale=15,
            linewidth=1.4, color="#333333",
            shrinkA=0, shrinkB=0,
            connectionstyle="arc3,rad=0.0",
            zorder=1,
        )
        ax.add_patch(arrow)

    plt.tight_layout()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return True


# ============================================================================
# 4. DRIVER
# ============================================================================
rng = random.Random(SAMPLE_SEED)
total_triples = 0

for scheme in SCHEME_TAGS:
    in_glob = os.path.join(DAG_DIR, f"*_parsed_{scheme}.jsonl")
    paths = sorted(glob.glob(in_glob))
    if not paths:
        print(f"[stage 6c] ({scheme}) no inputs matching {in_glob} -- skipping.")
        continue

    print(f"\n{'='*64}\n[stage 6c] scheme: {scheme}\n{'='*64}")
    for in_path in paths:
        dataset = os.path.basename(in_path).split("_parsed_")[0]
        recs = load_correct_consensus_records(in_path)
        weighted_idx = load_weighted_index(in_path, scheme)
        raw_cot_idx  = load_raw_cot_index(dataset)
        if not recs:
            print(f"  [{dataset}] no correct-consensus questions, skipping.")
            continue
        if not weighted_idx:
            print(f"  [{dataset}] no weighted DAG file, skipping.")
            continue
        if not raw_cot_idx:
            print(f"  [{dataset}] no raw CoT csv+jsonl pair, skipping.")
            continue

        k = min(N_QUESTIONS_PER_DATASET, len(recs))
        chosen = rng.sample(recs, k)
        out_dir = os.path.join(PNG_DIR, scheme, dataset)
        n_triples = n_skip = 0

        for rec in chosen:
            qid = rec.get("question_id", "unknown")
            cg = consensus_graph_from_record(rec)
            if cg is None or not cg.get("nodes"):
                n_skip += 1
                continue
            cg_nodeset = frozenset(cg["nodes"])

            wrec = weighted_idx.get(qid)
            if wrec is None:
                n_skip += 1
                continue
            ensemble = wrec.get("ensemble_dag") or {}
            id_to_node = {n["id"]: n for n in ensemble.get("nodes", [])}
            W = {n["id"]: float(n.get("W", 0.0))
                 for n in ensemble.get("nodes", [])}
            target = consensus_cluster_id(rec)
            if target is None or target not in id_to_node:
                n_skip += 1
                continue

            alt = get_alternative_graph(target, id_to_node, W, cg_nodeset,
                                        ALT_STRATEGY, rng)
            if alt is None or not alt.get("nodes"):
                n_skip += 1
                continue

            # raw CoT: a random trace from this question that reached
            # the CORRECT answer (== gold == consensus, since the question
            # came from the correct-consensus pool).
            cot_g = pick_random_correct_cot(
                raw_cot_idx.get(qid, []),
                rec.get("gold_label", ""),
                rng)
            if cot_g is None or not cot_g.get("nodes"):
                n_skip += 1
                continue

            cons_text = rec.get("consensus_answer", "?")
            gold_text = rec.get("gold_label", "?")
            base_hdr = (f"{dataset}  |  {qid}  |  weighting={scheme}\n"
                        f"conclusion = {cons_text}    gold = {gold_text}")
            cons_title = base_hdr + f"\n{len(cg['nodes'])} nodes  |  consensus graph"
            alt_title  = base_hdr + f"\n{len(alt['nodes'])} nodes  |  alternative graph"
            cot_src    = f"{cot_g.get('_source_model','?')} sample {cot_g.get('_sample_idx','?')}"
            cot_title  = base_hdr + f"\n{len(cot_g['nodes'])} nodes  |  raw CoT ({cot_src})"

            ok1 = render_graph(cg,    cons_title,
                               os.path.join(out_dir, f"{qid}_consensus.png"))
            ok2 = render_graph(alt,   alt_title,
                               os.path.join(out_dir, f"{qid}_alternative.png"))
            ok3 = render_graph(cot_g, cot_title,
                               os.path.join(out_dir, f"{qid}_rawcot.png"))

            if ok1 and ok2 and ok3:
                meta = {
                    "dataset": dataset, "question_id": qid, "scheme": scheme,
                    "consensus_answer": cons_text, "gold_label": gold_text,
                    "consensus": {
                        "n_nodes": len(cg["nodes"]),
                        "score":   round(sum(W.get(x, 0.0)
                                             for x in cg["nodes"]), 6),
                    },
                    "alternative": {
                        "n_nodes": len(alt["nodes"]),
                        "score":   round(sum(W.get(x, 0.0)
                                             for x in alt["nodes"]), 6),
                        "strategy": ALT_STRATEGY,
                        "node_set_diff_vs_consensus":
                            len(frozenset(cg["nodes"]).symmetric_difference(
                                frozenset(alt["nodes"]))),
                    },
                    "rawcot": {
                        "n_nodes":      len(cot_g["nodes"]),
                        "source_model": cot_g.get("_source_model"),
                        "sample_idx":   cot_g.get("_sample_idx"),
                    },
                }
                with open(os.path.join(out_dir, f"{qid}_meta.json"), "w") as f:
                    json.dump(meta, f, indent=2)
                n_triples += 1
            else:
                n_skip += 1

        total_triples += n_triples
        print(f"  [{dataset}] correct-consensus pool={len(recs)}, "
              f"sampled={k}, drew {n_triples} triples, skipped {n_skip}  "
              f"-> {out_dir}/")

print(f"\n[stage 6c] done. {total_triples} triples written under {PNG_DIR}/")
print(f"[stage 6c] layout:")
print(f"  {PNG_DIR}/<scheme>/<dataset>/<qid>_consensus.png")
print(f"  {PNG_DIR}/<scheme>/<dataset>/<qid>_alternative.png")
print(f"  {PNG_DIR}/<scheme>/<dataset>/<qid>_rawcot.png")
print(f"  {PNG_DIR}/<scheme>/<dataset>/<qid>_meta.json")

In [ ]:
# ============================================================================
# STAGE 6c-txt -- save the raw CoT text for the same traces picked in 6c
#
# Reuses the same SAMPLE_SEED and selection logic so this script picks the
# IDENTICAL trace per question as Stage 6c's _rawcot.png. The .txt file goes
# next to the PNGs and contains the unprocessed CoT (Stage 1's `reasoning`
# column from the eval CSV).
# ============================================================================

import csv, glob, json, os, random, sys
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

csv.field_size_limit(sys.maxsize)

DAG_DIR = "./outputs_4234"
PNG_DIR = os.path.join(DAG_DIR, "consensus_vs_alternative_vs_rawcot_pngs")

SCHEME_TAGS             = ["simple"]
N_QUESTIONS_PER_DATASET = 20
SAMPLE_SEED             = 7777


def load_correct_consensus_records(parsed_path: str) -> List[Dict]:
    out = []
    if not os.path.exists(parsed_path):
        return out
    with open(parsed_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            gold = (rec.get("gold_label") or "").strip().lower()
            cons = (rec.get("consensus_answer") or "").strip().lower()
            if gold and cons and gold == cons:
                out.append(rec)
    return out


def load_cot_text_index(dataset: str) -> Dict[str, List[Dict]]:
    """{question_id -> list of {model, sample_idx, predicted_label, reasoning,
                                raw_output}} read from the eval CSV.
    We also pull the DAG file just to skip traces where DAG extraction failed,
    so the selection matches Stage 6c exactly."""
    csv_path   = os.path.join(DAG_DIR, f"{dataset}_eval_cots.csv")
    jsonl_path = os.path.join(DAG_DIR, f"{dataset}_eval_dags.jsonl")
    if not (os.path.exists(csv_path) and os.path.exists(jsonl_path)):
        return {}

    # set of valid (qid, model, sample_idx) keys -- traces whose DAG parsed.
    valid_keys = set()
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not r.get("parse_ok") or r.get("dag") is None:
                continue
            try:
                valid_keys.add((r["question_id"], r["model"],
                                int(r["sample_idx"])))
            except (KeyError, ValueError, TypeError):
                continue

    by_qid: Dict[str, List[Dict]] = defaultdict(list)
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            qid   = (row.get("question_id") or "").strip()
            model = (row.get("model") or "").strip()
            try:
                samp = int(row.get("sample_idx") or "")
            except ValueError:
                continue
            if (qid, model, samp) not in valid_keys:
                continue
            by_qid[qid].append({
                "model":           model,
                "sample_idx":      samp,
                "predicted_label": (row.get("predicted_label") or "").strip(),
                "reasoning":       row.get("reasoning") or "",
                "raw_output":      row.get("raw_output") or "",
            })
    return dict(by_qid)


def pick_random_correct_cot(traces: List[Dict], gold_answer: str,
                             rng: random.Random) -> Optional[Dict]:
    want = (gold_answer or "").strip().lower()
    matching = [t for t in traces
                if (t.get("predicted_label") or "").strip().lower() == want]
    return rng.choice(matching) if matching else None


# ----------------------------------------------------------------------------
# Driver -- must replay Stage 6c's RNG calls in the same order so that the
# .rng.sample() and .rng.choice() picks line up. We don't call the alt-graph
# unfold here (no DAG access needed), but Stage 6c's `rng` was used for both
# question sampling AND the random fallbacks inside get_alternative_graph,
# AND for pick_random_correct_cot. To stay aligned we'd need to replay those
# too -- which requires the full pipeline.
#
# So instead: read each question's already-written meta.json (Stage 6c wrote
# it) which records the source_model + sample_idx of the chosen raw CoT.
# That guarantees the .txt matches the .png exactly regardless of RNG order.
# ----------------------------------------------------------------------------
total_txt = 0
for scheme in SCHEME_TAGS:
    in_glob = os.path.join(DAG_DIR, f"*_parsed_{scheme}.jsonl")
    paths = sorted(glob.glob(in_glob))
    for in_path in paths:
        dataset = os.path.basename(in_path).split("_parsed_")[0]
        out_dir = os.path.join(PNG_DIR, scheme, dataset)
        if not os.path.isdir(out_dir):
            continue

        cot_idx = load_cot_text_index(dataset)
        if not cot_idx:
            print(f"  [{dataset}] no CoT csv/jsonl, skipping.")
            continue

        n_done = n_skip = 0
        for meta_path in sorted(glob.glob(os.path.join(out_dir, "*_meta.json"))):
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
            qid    = meta["question_id"]
            model  = meta.get("rawcot", {}).get("source_model")
            samp   = meta.get("rawcot", {}).get("sample_idx")
            if model is None or samp is None:
                n_skip += 1
                continue

            # find the trace in the CSV index that matches the meta
            trace = next(
                (t for t in cot_idx.get(qid, [])
                 if t["model"] == model and t["sample_idx"] == samp),
                None)
            if trace is None:
                n_skip += 1
                continue

            txt_path = os.path.join(out_dir, f"{qid}_rawcot.txt")
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(f"dataset:         {dataset}\n")
                f.write(f"question_id:     {qid}\n")
                f.write(f"source_model:    {model}\n")
                f.write(f"sample_idx:      {samp}\n")
                f.write(f"predicted_label: {trace['predicted_label']}\n")
                f.write(f"gold_label:      {meta.get('gold_label','?')}\n")
                f.write(f"{'='*60}\n")
                f.write("REASONING (parsed from <reasoning>...</reasoning>):\n")
                f.write(f"{'='*60}\n")
                f.write(trace["reasoning"].rstrip() + "\n")
                f.write(f"\n{'='*60}\n")
                f.write("RAW MODEL OUTPUT (verbatim):\n")
                f.write(f"{'='*60}\n")
                f.write(trace["raw_output"].rstrip() + "\n")
            n_done += 1

        total_txt += n_done
        print(f"  [{dataset}] wrote {n_done} txt files "
              f"(skipped {n_skip}, missing meta or trace)  -> {out_dir}/")

print(f"\n[done] {total_txt} raw-cot .txt files written.")
print(f"layout: {PNG_DIR}/<scheme>/<dataset>/<qid>_rawcot.txt")

In [ ]:
import csv, glob, json, os, sys

csv.field_size_limit(sys.maxsize)

DAG_DIR = "./outputs_4234"
OUT_DIR = os.path.join(DAG_DIR, "question_txts")
os.makedirs(OUT_DIR, exist_ok=True)


def fmt_block(title, text):
    text = (text or "").strip()
    if not text:
        return ""
    return f"\n{'='*60}\n{title}\n{'='*60}\n{text}\n"


def format_question(row):
    dataset = (row.get("dataset") or "").strip()

    out = [
        f"dataset:     {dataset}",
        f"question_id: {row.get('question_id','')}",
        f"split:       {row.get('split','')}",
        f"gold_label:  {row.get('gold_label','')}",
    ]

    if dataset == "sara":
        out.append(fmt_block("STATUTE", row.get("statute")))
        out.append(fmt_block("CASE SCENARIO", row.get("case")))
        out.append(fmt_block("HYPOTHESIS", row.get("hypothesis")))

    elif dataset == "argkp":
        out.append(fmt_block("TOPIC", row.get("topic")))
        out.append(fmt_block("STANCE", row.get("stance")))

    elif dataset == "gpqa":
        out.append(fmt_block("QUESTION", row.get("question_text")))
        opts = []
        for L, c in [("A","option_a"),("B","option_b"),("C","option_c"),("D","option_d")]:
            if (row.get(c) or "").strip():
                opts.append(f"{L}. {row[c]}")
        out.append(fmt_block("OPTIONS", "\n".join(opts)))

    elif dataset in {"musr_mm", "musr_op", "musr_ta"}:
        out.append(fmt_block("NARRATIVE", row.get("narrative")))
        out.append(fmt_block("QUESTION", row.get("question_text")))

        choices_json = row.get("choices_json") or ""
        try:
            choices = json.loads(choices_json)
            choices = "\n".join(
                f"{chr(ord('A') + i)}. {x}" for i, x in enumerate(choices)
            )
        except Exception:
            choices = choices_json
        out.append(fmt_block("OPTIONS", choices))

    elif dataset == "folio":
        out.append(fmt_block("PREMISES", row.get("premises")))
        out.append(fmt_block("CONCLUSION", row.get("conclusion")))

    return "\n".join(x for x in out if x).rstrip() + "\n"


seen = set()
n_written = 0

for csv_path in sorted(glob.glob(os.path.join(DAG_DIR, "*_cots.csv"))):
    print(f"[read] {csv_path}")

    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            qid = (row.get("question_id") or "").strip()
            dataset = (row.get("dataset") or "").strip()
            split = (row.get("split") or "").strip()

            if not qid or not dataset:
                continue

            key = (dataset, split, qid)
            if key in seen:
                continue
            seen.add(key)

            dataset_dir = os.path.join(OUT_DIR, dataset, split)
            os.makedirs(dataset_dir, exist_ok=True)

            out_path = os.path.join(dataset_dir, f"{qid}_question.txt")
            with open(out_path, "w", encoding="utf-8") as out:
                out.write(format_question(row))

            n_written += 1

print(f"\n[done] wrote {n_written} question txt files")
print(f"layout: {OUT_DIR}/<dataset>/<split>/<qid>_question.txt")